# Simple RAG 구현

In [7]:
# !pip install llama-parse
# !pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 10.9 MB/s  0:00:00 eta 0:00:01


In [2]:
import os
from dotenv import load_dotenv
from llama_parse import LlamaParse
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser



/var/folders/sk/_9bn_nr9727cxqwf9sxykqxr0000gn/T/ipykernel_71437/44829664.py:3: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse
/opt/miniconda3/envs/llm_stable/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
# [강사님 코드 참고: 01_rag_workflow.ipynb - 환경변수 로드]
load_dotenv()

# 1. 문서 전처리 (수정됨: 최신 LlamaParse 문법 적용)
def load_and_parse_cigna(file_path):
    """
    [제미나이 전처리 추천 - 수정본]
    parsing_instruction 대신 user_prompt를 사용하여 표 추출 성능을 높였습니다.
    """
    parser = LlamaParse(
        result_type="markdown", 
        language="en",
        # 경고 메시지 해결을 위해 user_prompt로 변경
        user_prompt="This document is a Cigna Insurance guide. Please extract all benefit tables accurately, especially the 'Mental Health Care' section under 'International Outpatient'."
    )
    documents = parser.load_data(file_path)
    # 분석된 텍스트 합치기
    full_text = "\n\n".join([doc.text for doc in documents])
    return [full_text]

# 2. 인덱싱 (강사님 코드 참고: 02-1_indexing.ipynb)
def create_vector_db(texts):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    # 텍스트가 너무 길면 검색이 어려우므로 문서를 더 잘게 쪼개는 로직 추가 가능
    vector_db = FAISS.from_texts(texts, embeddings)
    return vector_db

# 3. RAG 체인 (강사님 코드 스타일 유지)
def build_cigna_rag_chain(vector_db):
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    # [핵심] LLM이 답변을 못 할 경우 '모른다'고 하지 말고 끝까지 찾게 유도하는 프롬프트
    template = """You are an expert in Cigna Global Health Insurance. 
    Using the provided context, find the specific coverage for 'Mental Health Care'.
    
    Check the 'International Outpatient' table in the context.
    If you find 'Mental Health Care', look for 'visit limits' or 'sessions'.
    
    Context:
    {context}
    
    Question: {question}
    
    Answer (in Korean):"""
    
    prompt = ChatPromptTemplate.from_template(template)

    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | model
        | StrOutputParser()
    )
    return chain

# 실행 로직은 동일

In [4]:
# --- 팀 시연용 실행 로직 ---
FILE_PATH = "CIGNA 591048 CGHO Customer Guide EN_05_2022 (1).pdf"

if __name__ == "__main__":
    print("🚀 Cigna 약관 분석 시작 (LlamaParse)...")
    texts = load_and_parse_cigna(FILE_PATH)
    
    print("📦 Vector DB 인덱싱 완료 (FAISS)...")
    vector_db = create_vector_db(texts)
    
    print("💬 RAG 시스템 가동...")
    rag_chain = build_cigna_rag_chain(vector_db)
    
    # 시연 질문
    query = "Cigna International Outpatient 플랜에서 심리상담(Mental Health Care) 보장 범위가 어떻게 돼?"
    
    print(f"\n[유저 질문]: {query}")
    response = rag_chain.invoke(query)
    print(f"\n[AI 분석 결과]:\n{response}")

🚀 Cigna 약관 분석 시작 (LlamaParse)...
Started parsing the file under job_id ce64cee1-b53b-4bd7-941e-ba2c6deed0b2
📦 Vector DB 인덱싱 완료 (FAISS)...
💬 RAG 시스템 가동...

[유저 질문]: Cigna International Outpatient 플랜에서 심리상담(Mental Health Care) 보장 범위가 어떻게 돼?

[AI 분석 결과]:
제공된 문서에는 'Cigna International Outpatient' 플랜의 심리상담(Mental Health Care) 보장 범위에 대한 정보가 포함되어 있지 않습니다. 'International Outpatient' 테이블에서 관련 내용을 찾을 수 없었습니다. 추가적인 정보가 필요하시면 Cigna의 공식 웹사이트나 고객 서비스에 문의하시기 바랍니다.
